# In this model we want to do the following:
- Model the white light curve for both visits
- Use fit results to fix the ramp and breathing parameters

In [ ]:
from preamble_jax import *

In [ ]:
visit = 'S22'

orbits_to_exclude = np.array([0,2])
err_factor = F21_SED_err_factor if visit=='F21' else S22_SED_err_factor
lc_index = 1 if visit=='F21' else 0
bin_edges = jnp.array([0.86,1.138,1.645]) * u.micron

predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']

rainbow = read_rainbow(f'../data/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy')
speclc_rainbow = rainbow
binned_for_model = rainbow.bin(wavelength_edges=bin_edges)
fleck_wavelengths = binned_for_model.wavelength.value

'Adjust uncertainties and bin the rainbow'
for i in range(len(rainbow.wavelength.value)):
    speclc_rainbow.uncertainty[i,:] = rainbow.uncertainty[i,:] * err_factor[i]

trimmed_rainbow = speclc_rainbow.get_average_lightcurve_as_rainbow()

trimmed_rainbow.time[-12:] = np.nan
# trimmed_rainbow.time[1:12] = np.nan

plt.figure(figsize=(9,3))
plt.errorbar(trimmed_rainbow.time,trimmed_rainbow.flux.flatten(),trimmed_rainbow.uncertainty.flatten(),
            fmt='o',ms=1)
plt.show()
plt.clf()

data_wavelengths = speclc_rainbow.wavelength.value
img_date = trimmed_rainbow.time.value
data_flux = trimmed_rainbow.flux.flatten().value
relative_err = trimmed_rainbow.uncertainty.flatten().value/data_flux
time_from_T0 = img_date - predicted_T0

plt.figure(figsize=(9,3) )
plt.errorbar(time_from_T0,data_flux,yerr=relative_err,fmt='o',ms=1)
plt.show()
plt.clf()

In [ ]:
# Label the orbits
orbit = np.zeros_like(img_date)
for j in range(len(img_date)):
    if j >= 1:
        if (img_date[j] - img_date[j - 1]) > 0.01:
            orbit[j] = (orbit[j - 1] + 1)
        else:
            orbit[j] = orbit[j - 1]

# Trim the first point from each orbit
ref_time = []
for o in np.unique(orbit):
    first_index = np.where(orbit == o)[0][0]
    ref_time.append(img_date[first_index])
    data_flux[first_index] = np.nan  # Set the first point of each subsequent orbit to np.nan
    relative_err[first_index] = np.nan
    img_date[first_index] = np.nan
    time_from_T0[first_index] = np.nan

# Set data to nan if it was in the pre-defined list of orbits to exclude
for orbit_to_exclude in orbits_to_exclude:
    data_flux[orbit == orbit_to_exclude] = np.nan
    relative_err[orbit == orbit_to_exclude] = np.nan
    img_date[orbit == orbit_to_exclude] = np.nan
    time_from_T0[orbit == orbit_to_exclude] = np.nan

plt.figure(figsize=(9,3) )
plt.errorbar(time_from_T0,data_flux,yerr=relative_err,fmt='o',ms=1)
plt.show()
plt.clf()

# Populate ramp_phase time arrays
phase_list=[]
for o in [0,1,2,3,4,5,6,7]:
    rphase = (img_date[orbit==o] - ref_time[o]) / 0.066
    phase_list.append(rphase)
ramp_phase = np.concatenate(phase_list)
breathing_phase = ( (img_date-ref_time[0]+0.02) / 0.066 ) % 1

# plt.figure(figsize=(9,3) )
# plt.scatter(ramp_phase,img_date)
# plt.show()
# plt.clf()

# plt.figure(figsize=(9,3) )
# plt.errorbar(breathing_phase,data_flux,yerr=relative_err,fmt='o',ms=1)
# plt.show()
# plt.clf()

In [ ]:
# Convert all arrays to JAX arrays
_data_flux = jnp.array( data_flux[~np.isnan(time_from_T0)] )
data_flux = _data_flux[~jnp.isnan(_data_flux)]
_relative_err = jnp.array(relative_err[~np.isnan(time_from_T0)] )
relative_err = _relative_err[~jnp.isnan(_data_flux)]
_img_date = jnp.array(img_date[~np.isnan(time_from_T0)] )
img_date = _img_date[~jnp.isnan(_data_flux)]
_time_from_T0 = jnp.array(time_from_T0[~np.isnan(time_from_T0)] )
time_from_T0 = _time_from_T0[~jnp.isnan(_data_flux)]
_ramp_phase = jnp.array(ramp_phase[~np.isnan(ramp_phase)] )
ramp_phase = _ramp_phase[~jnp.isnan(_data_flux)]
_breathing_phase = jnp.array(breathing_phase[~np.isnan(breathing_phase)] )
breathing_phase = _breathing_phase[~jnp.isnan(_data_flux)]

normalized_data_flux = data_flux/jnp.mean(data_flux)
normalized_relative_err = relative_err*normalized_data_flux

plt.figure(figsize=(9,3) )
plt.errorbar(time_from_T0,normalized_data_flux,yerr=normalized_relative_err,fmt='o',ms=1)
plt.show()
plt.clf()

In [ ]:
default_params={

    # Ramp Model Parameters
    'r_1': 18.1,
    'r_2': -6.7,
    'r_3': 0.0135,
    #Breathing params
    'b_1':0,
    'b_2':0,
    'b_3':0,
    'b_4':0,
    'HST_period':0.066,
    
    # Planet parameters
    'Mp':8.0,
    'planet_i':89.5,
    'P_orb':8.463,
    't_0':0.00055,
    'rp_rstar': 0.044,
    'a_rstar': 19.15,
    'ecc': 0.0,
    'u_1':0.35,
    'u_2':0.15,
    
    # Stellar parameters
    'log_g':4.5,
    'stellar_i':89.5,
    'P_rot':4.86,
    'Rs':0.8,
    'Ms':0.6,
    'metallicity':0.0,
    'log_fixedspot_radii':-1.6,
    'f_cool_unocculted':0.2,
    'f_cool_occulted':0.2,
    'T_unocculted': 3100,
    'T_occulted': 3500,
    'T_phot': 4200,
    'T_spot': 3550,
    'beta':0.25,
    'limb_spot_lon':-1.3,
    'limb_spot_lat':1.2,
    'limb_spot_rad':0.22,
    'chord_spot_lon_1':0.05,
    'chord_spot_lat_1':1.2,
    'chord_spot_rad_1':0.2,
    'chord_spot_lon_2':0.86,
    'chord_spot_lat_2':1.68,
    'chord_spot_rad_2':0.06,
}

In [ ]:
def numpyro_model():

    """
    Define the probabilistic model in NUMPYRO
    """
    params = {}

    # Instrument Parameters
    params['r_1'] = numpyro.sample('r_1', dist.Uniform(8.0, 20.0))
    params['r_2'] = numpyro.sample('r_2', dist.Uniform(-7.0, -6))
    params['r_3'] = numpyro.sample('r_3', dist.Uniform(0.005,0.03))

    # Instrument Parameters
    params['b_1'] = numpyro.sample('b_1', dist.Uniform(-0.1,0.1))
    params['b_2'] = numpyro.sample('b_2', dist.Uniform(-0.4,0.2))
    params['b_3'] = numpyro.sample('b_3', dist.Uniform(-0.2,0.5))
    params['b_4'] = numpyro.sample('b_4', dist.Uniform(-0.3,0.1))

    # Baseline Parameters
    params['limb_spot_lon'] = numpyro.sample('limb_spot_lon', dist.Uniform(-1.35, -1.25))
    params['limb_spot_lat'] = 1.2 #numpyro.sample('limb_spot_lat', dist.Uniform(0.9, 1.3))
    params['limb_spot_rad'] = numpyro.sample('limb_spot_rad', dist.Uniform(0.1, 0.6))
    
    params['chord_spot_lon_1'] = numpyro.sample('chord_spot_lon_1', dist.Uniform(0.045, 0.09))
    params['chord_spot_lat_1'] = numpyro.sample('chord_spot_lat_1', dist.Uniform(1.51, 1.66))
    params['chord_spot_rad_1'] = numpyro.sample('chord_spot_rad_1', dist.Uniform(0.135, 0.27))

    params['chord_spot_lon_2'] = numpyro.sample('chord_spot_lon_2', dist.Uniform(0.825,0.895))
    params['chord_spot_lat_2'] = numpyro.sample('chord_spot_lat_2', dist.Uniform(1.65, 1.73))
    params['chord_spot_rad_2'] = numpyro.sample('chord_spot_rad_2', dist.Uniform(0.045, 0.1))

    # Planet Parameters
    params['rp_rstar'] = numpyro.sample('rp_rstar', dist.Uniform(0.043, 0.05))
    params['t_0'] = 0.00055#numpyro.sample('t_0', dist.Uniform(0.0, 0.001)) 
    params['a_rstar'] = 19.15 #numpyro.sample('a', dist.Uniform(18.0, 20.0))
    
    params['u_1'] = numpyro.sample('u_1', dist.Uniform(0.1,0.6))
    params['u_2'] = numpyro.sample('u_2', dist.Uniform(0.1,0.6))
    
    # Stellar Parameters
    params['T_phot'] = 4200
    params['T_spot'] = 3550 #numpyro.sample('T_spot', dist.Uniform(3400,3600))
    beta = numpyro.sample('beta', dist.Uniform(0.0, 0.35))

    @jit
    def broadband_transit(params,**kwargs):

        # Calculate systematics model
        ramp = ramp_model_jax(phase=ramp_phase,
                        r1=jnp.asarray(params['r_1'], dtype=jnp.float64),
                        r2=jnp.asarray(params['r_2'], dtype=jnp.float64),
                        r3=jnp.asarray(params['r_3'], dtype=jnp.float64) )
        breathing = breathing_model_jax(phase=breathing_phase,
                                  b1=jnp.asarray(params['b_1'], dtype=jnp.float64),
                                  b2=jnp.asarray(params['b_2'], dtype=jnp.float64),
                                  b3=jnp.asarray(params['b_3'], dtype=jnp.float64),
                                  b4=jnp.asarray(params['b_4'], dtype=jnp.float64))
        _systematics = (breathing * ramp)
        systematics = _systematics/jnp.mean(_systematics)
        
        # Update Planet Parameters
        planet_parameters = dict(
            inclination = jnp.radians(89.5),
            a = jnp.asarray(params['a_rstar'], dtype=jnp.float64),
            rp = jnp.asarray(params['rp_rstar'], dtype=jnp.float64),
            period = 8.46,
            t0 = jnp.asarray(params['t_0'], dtype=jnp.float64),
            ecc = 0.0,
            u1 = jnp.asarray(params['u_1'], dtype=jnp.float64),
            u2 = jnp.asarray(params['u_2'], dtype=jnp.float64))
        
        # Get spectra for the model temps
        Phot = get_binned_BTSettl_spectrum_jax(T=params['T_phot'],data_wave = fleck_wavelengths)
        Cool = get_binned_BTSettl_spectrum_jax(T=params['T_spot'],data_wave = fleck_wavelengths)
        
        # Update stellar parameters
        active_star = ActiveStar(
        times = time_from_T0,
        inclination=jnp.radians(89.5),
        T_eff=jnp.asarray(params['T_phot'], dtype=jnp.float64),
        wavelength=Phot[0]*1e-6, # m
        phot=Phot[1],
        P_rot=4.863)
        
        active_star.spectrum = jnp.concatenate([
            jnp.vstack([Cool[1],
                        Cool[1],
                        Cool[1]
                       ]),
        ])
        active_star.temperature = jnp.concatenate([
            jnp.array([params['T_spot'],
                       params['T_spot'],
                       params['T_spot']
                      ]),
        ])
        active_star.lon = jnp.concatenate([
            jnp.array([params['limb_spot_lon'],
                       params['chord_spot_lon_1'],
                       params['chord_spot_lon_2']
                      ]),
        ])
        active_star.lat = jnp.concatenate([
            jnp.array([params['limb_spot_lat'],
                       params['chord_spot_lat_1'], 
                       params['chord_spot_lat_2']
                      ]),
        ])
        active_star.rad = jnp.concatenate([
            jnp.array([params['limb_spot_rad'], 
                       params['chord_spot_rad_1'], 
                       params['chord_spot_rad_2']
                      ]),
        ])        
        
        lc, contam, X, Y, spectrum_at_transit = active_star.transit_model(**planet_parameters)        
        lc_model = lc.T[lc_index] * systematics
        mean_per_wavelength = jnp.mean(lc_model)
        normalized_model = lc_model / mean_per_wavelength
    
        return normalized_model

    numpyro.sample(
        'Obs', dist.Normal(
            loc=broadband_transit(params), 
            scale=(10**beta)*normalized_relative_err,
        ), obs=normalized_data_flux
    )

In [ ]:
def plot_results(parameter_dict):
    params = parameter_dict

    # Calculate systematics model
    ramp = ramp_model_jax(phase=ramp_phase,
                    r1=jnp.asarray(params.get('r_1',default_params['r_1']), dtype=jnp.float64),
                    r2=jnp.asarray(params.get('r_2',default_params['r_2']), dtype=jnp.float64),
                    r3=jnp.asarray(params.get('r_3',default_params['r_3']), dtype=jnp.float64) )
    breathing = breathing_model_jax(phase=breathing_phase,
                              b1=jnp.asarray(params.get('b_1',default_params['b_1']), dtype=jnp.float64),
                              b2=jnp.asarray(params.get('b_2',default_params['b_2']), dtype=jnp.float64),
                              b3=jnp.asarray(params.get('b_3',default_params['b_3']), dtype=jnp.float64),
                              b4=jnp.asarray(params.get('b_4',default_params['b_4']), dtype=jnp.float64))
    _systematics = (breathing * ramp)
    systematics = _systematics/jnp.mean(_systematics)
    
    # Update Planet Parameters
    planet_parameters = dict(
        inclination = jnp.radians(89.5),
        a = jnp.asarray(params.get('a_rstar',default_params['a_rstar']), dtype=jnp.float64),
        rp = jnp.asarray(params.get('rp_rstar',default_params['rp_rstar']), dtype=jnp.float64),
        period = 8.46,
        t0 = jnp.asarray(params.get('t_0',default_params['t_0']), dtype=jnp.float64),
        ecc = 0.0,
        u1 = jnp.asarray(params.get('u_1',default_params['u_1']), dtype=jnp.float64),
        u2 = jnp.asarray(params.get('u_2',default_params['u_2']), dtype=jnp.float64),
    )
    
    # Get spectra for the model temps
    Phot = get_binned_BTSettl_spectrum_jax(T=jnp.asarray(params.get('T_phot',default_params['T_phot']), dtype=jnp.float64),
                                           data_wave = fleck_wavelengths)
    Cool = get_binned_BTSettl_spectrum_jax(T=jnp.asarray(params.get('T_spot',default_params['T_spot']), dtype=jnp.float64),
                                           data_wave = fleck_wavelengths)
    
    # Update stellar parameters
    active_star = ActiveStar(
    times = time_from_T0,
    inclination=jnp.radians(89.5),
    T_eff=jnp.asarray(params.get('T_phot',default_params['T_phot']), dtype=jnp.float64),
    wavelength=Phot[0]*1e-6, # m
    phot=Phot[1],
    P_rot=4.863)
    
    active_star.spectrum = jnp.concatenate([
        jnp.vstack([Cool[1],Cool[1],Cool[1]
                   ]),
    ])
    active_star.temperature = jnp.concatenate([
        jnp.array([params.get('T_spot', default_params['T_spot']),
                   params.get('T_spot', default_params['T_spot']),
                   params.get('T_spot', default_params['T_spot'])
                  ]),
    ])
    active_star.lon = jnp.concatenate([
        jnp.array([params.get('limb_spot_lon', default_params['limb_spot_lon']), 
                   params.get('chord_spot_lon_1', default_params['chord_spot_lon_1']), 
                   params.get('chord_spot_lon_2', default_params['chord_spot_lon_2']),
                   ]),
    ])
    active_star.lat = jnp.concatenate([
        jnp.array([params.get('limb_spot_lat', default_params['limb_spot_lat']), 
                   params.get('chord_spot_lat_1', default_params['chord_spot_lat_1']), 
                   params.get('chord_spot_lat_2', default_params['chord_spot_lat_2']),
                   ]),
    ])
    active_star.rad = jnp.concatenate([
        jnp.array([params.get('limb_spot_rad', default_params['limb_spot_rad']), 
                   params.get('chord_spot_rad_1', default_params['chord_spot_rad_1']), 
                   params.get('chord_spot_rad_2', default_params['chord_spot_rad_2']),
                   ]),
    ])        
    
    lc, contam, X, Y, spectrum_at_transit = active_star.transit_model(**planet_parameters)        
    lc_model = lc.T[lc_index] * systematics
    mean_per_wavelength = jnp.mean(lc_model)
    normalized_model = lc_model / mean_per_wavelength

    return normalized_model,active_star

In [ ]:
rng_seed = 0

# def hstack_recursive(final_states, checkpoint_states):
#     for key in final_states.keys():
#         if isinstance(final_states[key], dict):
#             hstack_recursive(final_states[key], checkpoint_states[key])
#         else:
#             final_states[key] = jnp.hstack([
#                 final_states[key], 
#                 checkpoint_states[key]
#             ])
def hstack_recursive(final_states, checkpoint_states):
    for key in final_states.keys():
        if isinstance(final_states[key], dict):
            hstack_recursive(final_states[key], checkpoint_states[key])
        else:
            final_states[key] = jnp.concatenate([
                final_states[key], 
                checkpoint_states[key]
            ], axis=1)
            
def print_big_message(big_message):
    print('\n\n')
    print('=' * len(big_message))
    print(big_message)
    print('=' * len(big_message))
    print('\n\n')

In [ ]:
class MCMCWithCheckpoints(MCMC):
    running_states = None
    checkpoint = 0
    start_time = None
    n_checkpoints = None
    last_checkpoint = None

    def run_checkpoints(self, rng_key, *args, extra_fields=(), n_checkpoints=10, 
                        progress_bar_warmup=True, progress_bar_samples=True, 
                        init_params=None, on_checkpoint=None, **kwargs):
        """
        Run the MCMC samplers and collect samples.

        :param random.PRNGKey rng_key: Random number generator key to be used for the sampling.
            For multi-chains, a batch of `num_chains` keys can be supplied. If `rng_key`
            does not have batch_size, it will be split in to a batch of `num_chains` keys.
        :param args: Arguments to be provided to the :meth:`numpyro.infer.mcmc.MCMCKernel.init` method.
            These are typically the arguments needed by the `model`.
        :param extra_fields: Extra fields (aside from `"z"`, `"diverging"`) from the
            state object (e.g. :data:`numpyro.infer.hmc.HMCState` for HMC) to be collected
            during the MCMC run. Note that subfields can be accessed using dots, e.g.
            `"adapt_state.step_size"` can be used to collect step sizes at each step. Exclude sample sites from
            collection with "~`sampler.sample_field`.`sample_site`". e.g. "~z.a" will prevent site "a" from
            being collected if you're using the NUTS sampler. To collect samples of a site "a" in the
            unconstrained space, we can specify the variable here, e.g. `extra_fields=("z.a",)`.
        :type extra_fields: tuple or list of str
        :param init_params: Initial parameters to begin sampling. The type must be consistent
            with the input type to `potential_fn` provided to the kernel. If the kernel is
            instantiated by a numpyro model, the initial parameters here correspond to latent
            values in unconstrained space.
        :param kwargs: Keyword arguments to be provided to the :meth:`numpyro.infer.mcmc.MCMCKernel.init`
            method. These are typically the keyword arguments needed by the `model`.

        .. note:: jax allows python code to continue even when the compiled code has not finished yet.
            This can cause troubles when trying to profile the code for speed.
            See https://jax.readthedocs.io/en/latest/async_dispatch.html and
            https://jax.readthedocs.io/en/latest/profiling.html for pointers on profiling jax programs.
        """
        self.start_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
        num_warmup_total = int(self.num_warmup)
        num_samples_total = int(self.num_samples)
        
        check_point_indices = [
            jnp.arange(num_warmup_total), 
            *jnp.array_split(jnp.arange(num_samples_total), n_checkpoints)
        ]
        self.n_checkpoints = n_checkpoints
        rng_keys = random.split(rng_key, len(check_point_indices))
        pbar = tqdm(enumerate(zip(rng_keys, check_point_indices)), total=n_checkpoints)
        for checkpoint, (rng_key, bounds) in pbar:
            self.checkpoint = checkpoint
            if checkpoint == 0:
                self.progress_bar = progress_bar_warmup
                pbar.set_description('Run warmup')
                print_big_message("Begin warmup")
                self.warmup(rng_key, *args, extra_fields=extra_fields, init_params=init_params, **kwargs)
                print_big_message(f"Begin {num_samples_total} samples with {n_checkpoints} checkpoints")
                block_until_ready(self._warmup_state)
            else:
                self.last_checkpoint = datetime.now()
                pbar.set_description(f'Run samples {bounds.min()} to {bounds.max()}')

                self.progress_bar = progress_bar_samples
                self.num_samples = bounds.size
                self.run(rng_key, *args, extra_fields=extra_fields, init_params=init_params, **kwargs)
                block_until_ready(self._states)
                # add to running states:
                if self.running_states is None:
                    self.running_states = dict(self._states)
                else:
                    hstack_recursive(self.running_states, self._states)
                
                # ensure that calls to `self.get_samples` will build a new samples array
                # out of the running states:
                self._states = self.running_states
                self._states_flat = None

                if on_checkpoint is not None:
                    on_checkpoint(self, **kwargs)
        
        pbar.close()

        # reset to total number for arviz IO
        self.num_samples = num_samples_total

def post_batch_viz_save(self, **kwargs):
    """
    here we define some tasks to do after each completed checkpoint:
    """
    print(f'Corner for checkpoint {self.checkpoint}')
    samples_cumulative = self.get_samples()
    corner.corner(samples_cumulative)
    plt.suptitle(f'checkpoint {self.checkpoint}')
    plt.savefig(f'../figs/chkpt_{self.checkpoint}_corner.png',dpi=200)
    plt.show()
    plt.clf()
    
    result = arviz.from_numpyro(self)
    display(arviz.summary(result))

    median_params = arviz.summary(result,stat_focus='median')['median']

    checkpoint_model, checkpoint_star = plot_results(median_params)

    checkpoint_star.plot_star(
        t0=median_params.get('t_0', default_params['t_0']),
        rp=median_params.get('rp_rstar', default_params['rp_rstar']),  # Exoplanet radius in units of stellar radii
        a=median_params.get('a_rstar', default_params['a_rstar']),  # Planetary semi-major axis in units of stellar radii
        inclination=np.radians(89.5),  # Planetary orbital inclination [radians]
        ecc=median_params.get('ecc', default_params['ecc']),
    )
    
    plt.figure(figsize=(9,3) )
    plt.errorbar(time_from_T0,normalized_data_flux,yerr=(10**median_params.get('beta', default_params['beta']))*normalized_relative_err,fmt='o',ms=1,color='k')
    plt.scatter(time_from_T0,checkpoint_model,s=1,color='r')    
    plt.title(f'checkpoint {self.checkpoint} Plot')
    plt.savefig(f'../figs/chkpt_{self.checkpoint}_model.png',dpi=200)
    plt.show()
    plt.clf()

    with open(f'../data/samples/samples_cumulative_{self.start_time}_checkpoint_{self.checkpoint:04d}.pkl', 'wb') as file:
        pickle.dump(dict(samples_cumulative), file)

In [ ]:
n_temps = 2
n_spots = 3
n_warmup = 1_00
n_samples = 1_000
n_chains = 7
n_check=10

model_designation = f'{visit}_{n_samples}samples_{n_chains}chains_testing_plotting_function'

rng_key = PRNGKey(22)

sampler = NUTS(
    numpyro_model,
    dense_mass=True
)
mcmc = MCMCWithCheckpoints(
    sampler, 
    num_warmup=n_warmup, 
    num_samples=n_samples,
    num_chains=n_chains
)

mcmc.run_checkpoints(rng_key, n_checkpoints=n_check, on_checkpoint=post_batch_viz_save)

mcmc.print_summary()                                                                        

# arviz converts a numpyro MCMC object to an `InferenceData` object based on xarray:
result = arviz.from_numpyro(mcmc)

In [ ]:
'Save the result'
result.to_netcdf(f'../data/samples/{model_designation}')
'Print the summary'
arviz.summary(result)

In [ ]:
'Make a corner plot'
corner.corner(
    result, 
    # var_names=['~offset_G102',
    #                    r'~$\beta_{\rm G102}$',
    #                    r'~$\beta_{\rm G141}$',
    #                    r'~$\sigma_{\rm conv}$'],
    # quiet=True, 
);
plt.savefig(f'../figs/{model_designation}_corner.png',dpi=200)

median_params = arviz.summary(result,stat_focus='median')['median']

'Examine the Leave-One-Out (LOO) summary'
loo = arviz.loo(result, pointwise=True)
loo

In [ ]:
all_vars = result.groups()
print("Available groups:", all_vars)

if 'posterior' in all_vars:
    posterior_vars = list(result.posterior.data_vars.keys())
    print("Posterior parameters:", posterior_vars)
    
    map_estimates = {}
    for param in posterior_vars:
        # Flatten all chains and draws
        samples = result.posterior[param].values.flatten()
        
        # Use KDE to find mode (true MAP)
        kde = gaussian_kde(samples)
        n_points = 1000
        x = np.linspace(samples.min(), samples.max(), n_points)
        density = kde(x)
        map_value = x[density.argmax()]
        map_estimates[param] = map_value
        
        print(f"{param}: {map_value:.4f}")

In [ ]:
mle_params = {'b_1': -0.0095,
              'b_2': -0.0364,
              'b_3': 0.0602,
              'b_4': -0.0308,
              'chord_spot_1_lat': 1.5576,
              'chord_spot_1_lon': 0.0682,
              'chord_spot_1_rad': 0.1461,
              'chord_spot_2_lat': 1.6585,
              'chord_spot_2_lon': 0.8496,
              'chord_spot_2_rad': 0.0729,
              'limb_spot_lat': 1.2120,
              'limb_spot_lon': -1.303,
              'limb_spot_rad': 0.2794,
              'r_1': 12.0379,
              'r_2': -6.6870,
              'r_3': 0.015,
              'rp_rstar': 0.0489,
              'u_1': 0.3663,
              'u_2': 0.308
}

In [ ]:
model, plottable_star = plot_results(mle_params)

plottable_star.plot_star(
    t0=0.00055,#median_params['t_0'],
    rp=mle_params['rp_rstar'],  # Exoplanet radius in units of stellar radii
    a=19.15,  # Planetary semi-major axis in units of stellar radii
    inclination=np.radians(89.5),  # Planetary orbital inclination [radians]
    ecc=0)

plt.figure(figsize=(9,3) )
plt.errorbar(time_from_T0,normalized_data_flux,yerr=(10**0.25)*normalized_relative_err,fmt='o',ms=1,color='k')
plt.scatter(time_from_T0,model,s=1,color='r')
plt.show()
plt.clf()